In [ ]:
"""
Cell 0 — Setup, Shared Constants, Overall Alpha & Agreement

This cell must be run first. It sets up everything that all subsequent
cells depend on.

Each annotator's Excel file must contain exactly three columns:
  Jokes | Technique | Rating

Steps:
  1. FILE PATHS         : Define paths to the three annotators' Excel files.
  2. LOAD FILES         : Read all three files into DataFrames (df1, df2, df3).
                          Strip whitespace and uppercase both label columns to
                          avoid case/spacing mismatches.
  3. MERGE              : Inner-join all three DataFrames on 'Jokes', producing
                          a single wide DataFrame ('merged') with columns:
                          Jokes, Tech_A1, Tech_A2, Tech_A3,
                                 Rate_A1, Rate_A2, Rate_A3.
  SHARED CONSTANTS      : Define tech_labels, rate_labels, annotators, and
                          pairs — used by all later cells.
  4. KRIPPENDORFF'S ALPHA : Compute the three-way nominal alpha for Technique
                          and ordinal alpha for Rating.
  5. PAIRWISE AGREEMENT : Raw count and % agreement for each annotator pair,
                          for both Technique and Rating.
  6. FULL AGREEMENT TABLE : Per-joke summary of all six labels and boolean
                          full-agreement flags for each label set.
  7. SAVE OUTPUT        : Write the full agreement table to Excel.

Produces:
  merged          — shared DataFrame used by all later cells
  tech_labels     — list of Technique label strings ['A','I','M','S','W']
  rate_labels     — list of Rating label values [0, 1]
  annotators      — list ['A1', 'A2', 'A3']
  pairs           — list of annotator pair tuples
  agreement_df    — per-joke agreement table
  agreement_table.xlsx
"""

# ============================================================
# Krippendorff's Alpha + Inter-Annotator Agreement Analysis
# For 3 Excel files containing joke annotations
# Columns: Jokes, Technique, Rating
# ============================================================

# Install if needed:
# pip install pandas openpyxl krippendorff

import pandas as pd
import numpy as np
import krippendorff

# ============================================================
# STEP 1: FILE PATHS
# ============================================================

file1 = "./annotation_data/pilot/annotator_1_pilot.xlsx"
file2 = "./annotation_data/pilot/annotator_2_pilot.xlsx"
file3 = "./annotation_data/pilot/annotator_3_pilot.xlsx"

# ============================================================
# STEP 2: LOAD FILES
# ============================================================

df1 = pd.read_excel(file1)
df2 = pd.read_excel(file2)
df3 = pd.read_excel(file3)

# Ensure correct column names
df1.columns = ['Jokes', 'Technique', 'Rating']
df2.columns = ['Jokes', 'Technique', 'Rating']
df3.columns = ['Jokes', 'Technique', 'Rating']

# Strip whitespace and uppercase Technique labels to fix spacing mismatches
df1['Technique'] = df1['Technique'].str.strip().str.upper()
df2['Technique'] = df2['Technique'].str.strip().str.upper()
df3['Technique'] = df3['Technique'].str.strip().str.upper()

# Ensure Rating is integer
df1['Rating'] = df1['Rating'].fillna(0).astype(int)
df2['Rating'] = df2['Rating'].fillna(0).astype(int)
df3['Rating'] = df3['Rating'].fillna(0).astype(int)

# ============================================================
# STEP 3: MERGE DATA PROPERLY (by Jokes)
# ============================================================

merged = df1.rename(columns={'Technique': 'Tech_A1', 'Rating': 'Rate_A1'})

merged = merged.merge(
    df2.rename(columns={'Technique': 'Tech_A2', 'Rating': 'Rate_A2'}),
    on='Jokes'
)

merged = merged.merge(
    df3.rename(columns={'Technique': 'Tech_A3', 'Rating': 'Rate_A3'}),
    on='Jokes'
)

# ============================================================
# SHARED CONSTANTS
# ============================================================

tech_labels = ['A', 'I', 'M', 'S', 'W']
rate_labels = [0, 1]
annotators  = ['A1', 'A2', 'A3']
pairs       = [('A1','A2'), ('A1','A3'), ('A2','A3')]

# ============================================================
# STEP 4: KRIPPENDORFF'S ALPHA
# ============================================================

print("\n==============================")
print("KRIPPENDORFF'S ALPHA")
print("==============================")

tech_data = [
    merged['Tech_A1'].tolist(),
    merged['Tech_A2'].tolist(),
    merged['Tech_A3'].tolist()
]

rate_data = [
    merged['Rate_A1'].tolist(),
    merged['Rate_A2'].tolist(),
    merged['Rate_A3'].tolist()
]

alpha_tech = krippendorff.alpha(
    reliability_data=tech_data,
    level_of_measurement='nominal'
)

alpha_rate = krippendorff.alpha(
    reliability_data=rate_data,
    level_of_measurement='ordinal'
)

print(f"Technique Alpha : {round(alpha_tech, 4)}")
print(f"Rating Alpha    : {round(alpha_rate, 4)}")

# ============================================================
# STEP 5: PAIRWISE AGREEMENT
# ============================================================

print("\n==============================")
print("PAIRWISE AGREEMENT")
print("==============================")

for p1, p2 in pairs:

    tech_col1 = f'Tech_{p1}'
    tech_col2 = f'Tech_{p2}'
    rate_col1 = f'Rate_{p1}'
    rate_col2 = f'Rate_{p2}'

    total = len(merged)

    tech_agree = (merged[tech_col1] == merged[tech_col2]).sum()
    rate_agree = (merged[rate_col1] == merged[rate_col2]).sum()

    tech_pct = tech_agree / total * 100
    rate_pct = rate_agree / total * 100

    print(f"\nPair {p1}-{p2}:")
    print(f"  Technique Agreement : {tech_agree}/{total} = {tech_pct:.2f}%")
    print(f"  Rating Agreement    : {rate_agree}/{total} = {rate_pct:.2f}%")

# ============================================================
# STEP 6: FULL AGREEMENT TABLE
# ============================================================
label_info={'A':'Absurdity', 'W':'Wordplay', 'S':'Surprise', 'I':'Irony', 'M':'Miscellaneous'}
print(label_info)
print("\n==============================")
print("FULL AGREEMENT TABLE")
print("==============================")

agreement_rows = []

for _, row in merged.iterrows():

    tech_vals = [row['Tech_A1'], row['Tech_A2'], row['Tech_A3']]
    rate_vals = [row['Rate_A1'], row['Rate_A2'], row['Rate_A3']]

    tech_agree = len(set(tech_vals)) == 1
    rate_agree = len(set(rate_vals)) == 1

    agreement_rows.append([
        row['Jokes'],
        tech_vals,
        tech_agree,
        rate_vals,
        rate_agree
    ])

agreement_df = pd.DataFrame(
    agreement_rows,
    columns=[
        'Jokes',
        'Technique Labels',
        'Technique Full Agreement',
        'Rating Labels',
        'Rating Full Agreement'
    ]
)

print(agreement_df)

# ============================================================
# STEP 7: SAVE OUTPUT
# ============================================================

output_path =  "./data_analysis/agreement_table_for_pilot.xlsx"


agreement_df.to_excel(output_path, index=False)

print(f"\nSaved file at: {output_path}")



KRIPPENDORFF'S ALPHA
Technique Alpha : 0.417
Rating Alpha    : 0.0602
{'A': 'Absurdity', 'W': 'Wordplay', 'S': 'Surprise', 'I': 'Irony', 'M': 'Miscellaneous'}

PAIRWISE AGREEMENT

Pair A1-A2:
  Technique Agreement : 115/147 = 78.23%
  Rating Agreement    : 83/147 = 56.46%

Pair A1-A3:
  Technique Agreement : 87/147 = 59.18%
  Rating Agreement    : 76/147 = 51.70%

Pair A2-A3:
  Technique Agreement : 92/147 = 62.59%
  Rating Agreement    : 80/147 = 54.42%

FULL AGREEMENT TABLE
                                                 Jokes Technique Labels  \
0    What does Hitler call the area around his tent...        [W, W, W]   
1    Why couldn’t the poor man donate at the sperm ...        [W, W, W]   
2    I identify as an elongated fish._____People sa...        [W, W, W]   
3    Crazy ex-girlfriends are like a box of chocola...        [S, S, S]   
4    How did Metallica get people to stop pirating ...        [S, S, I]   
..                                                 ...              

In [3]:
"""
Cell 1 — Pairwise Krippendorff's Alpha Between Annotators

Depends on: merged, pairs (defined in Cell 0).

Computes Krippendorff's Alpha separately for each of the three annotator
pairs (A1-A2, A1-A3, A2-A3), for both Technique (nominal) and Rating
(ordinal). This isolates which specific pair of annotators agrees most
and which diverges most, for each label set independently.

Steps:
  pairwise_alpha()  : Helper that takes two column names from 'merged'
                      and a level_of_measurement string, and returns the
                      pairwise Krippendorff's Alpha rounded to 4 d.p.
  PAIRS LOOP        : Calls pairwise_alpha() for each annotator pair on
                      both Technique and Rating columns, and prints results.
  SAVE              : Writes all pairwise alpha scores to an Excel file.

Produces:
  result_df                    — DataFrame with columns:
                                 Annotator 1, Annotator 2,
                                 Technique Alpha, Rating Alpha
  pairwise_krippendorff_alpha.xlsx
"""

# ============================================================
# PAIRWISE KRIPPENDORFF'S ALPHA BETWEEN ANNOTATORS
# Technique (nominal) + Rating (ordinal)
# ============================================================

# merged, pairs already defined in Cell 0

import krippendorff

# ------------------------------------------------------------
# FUNCTION: PAIRWISE KRIPPENDORFF'S ALPHA
# ------------------------------------------------------------

def pairwise_alpha(col1, col2, level):

    data = [
        merged[col1].tolist(),
        merged[col2].tolist()
    ]

    alpha = krippendorff.alpha(
        reliability_data=data,
        level_of_measurement=level
    )

    return round(alpha, 4)

# ------------------------------------------------------------
# PAIRS LOOP
# ------------------------------------------------------------

print("\n====================================")
print("PAIRWISE KRIPPENDORFF'S ALPHA")
print("====================================")

results = []

for a, b in pairs:

    tech_alpha = pairwise_alpha(
        f"Tech_{a}", f"Tech_{b}", 'nominal'
    )

    rate_alpha = pairwise_alpha(
        f"Rate_{a}", f"Rate_{b}", 'ordinal'
    )

    print(f"\nPair: {a} vs {b}")
    print(f"  Technique Alpha = {tech_alpha}")
    print(f"  Rating Alpha    = {rate_alpha}")

    results.append([a, b, tech_alpha, rate_alpha])

# ------------------------------------------------------------
# SAVE RESULTS
# ------------------------------------------------------------

result_df = pd.DataFrame(
    results,
    columns=[
        "Annotator 1",
        "Annotator 2",
        "Technique Alpha",
        "Rating Alpha"
    ]
)

output_path =  "./data_analysis/pairwise_krippendorff_alpha_pilot.xlsx"

result_df.to_excel(output_path, index=False)

print(f"\nSaved: {output_path}")



PAIRWISE KRIPPENDORFF'S ALPHA

Pair: A1 vs A2
  Technique Alpha = 0.5996
  Rating Alpha    = 0.1231

Pair: A1 vs A3
  Technique Alpha = 0.3178
  Rating Alpha    = 0.0062

Pair: A2 vs A3
  Technique Alpha = 0.3383
  Rating Alpha    = 0.0463

Saved: ./data_analysis/pairwise_krippendorff_alpha_pilot.xlsx


In [ ]:
"""
Cell 2 — Extended Label-Level Agreement Analysis

Depends on: merged, tech_labels, rate_labels, annotators, pairs
            (defined in Cell 0).

Runs all four parts of the extended analysis in parallel for both
Technique and Rating label sets.

Parts:
  1. OVERALL AGREEMENT / DISAGREEMENT
       For each label in Technique and each label in Rating, counts how
       many times at least 2 of the 3 annotators agreed on that label
       (agreement) vs exactly 1 used it (disagreement). Highlights the
       label with the most agreement and most disagreement per set.

  2. PAIRWISE LABEL AGREEMENT / DISAGREEMENT
       Repeats the above for each of the three annotator pairs
       individually, for both Technique and Rating.

  3. LABEL RATIOS PER ANNOTATOR
       Per-annotator percentage share of each label, for both label sets.

  4. AVERAGE RATIOS ACROSS ALL ANNOTATORS
       Pooled percentage share of each label across all three annotators,
       for both label sets.
"""

# ============================================================
# EXTRA ANALYSIS — Technique + Rating
# ============================================================

# merged, tech_labels, rate_labels, annotators, pairs
# already defined in Cell 0

import pandas as pd

# ============================================================
# PART 1: OVERALL AGREEMENT / DISAGREEMENT
# ============================================================
label_info={'A':'Absurdity', 'W':'Wordplay', 'S':'Surprise', 'I':'Irony', 'M':'Miscellaneous'}
print(label_info)
print("\n================================================")
print("OVERALL LABEL AGREEMENT / DISAGREEMENT")
print("================================================")

# ── Technique ────────────────────────────────────────────────
print("\n── Technique ──")

tech_overall = []

for label in tech_labels:

    agreement    = 0
    disagreement = 0

    for _, row in merged.iterrows():

        vals  = [row["Tech_A1"], row["Tech_A2"], row["Tech_A3"]]
        count = vals.count(label)

        if count >= 2:
            agreement += 1
        elif count == 1:
            disagreement += 1

    tech_overall.append([label, agreement, disagreement])

tech_result = pd.DataFrame(
    tech_overall,
    columns=["Technique", "Agreement", "Disagreement"]
)

print(tech_result)
print("\nMOST AGREEMENT:")
print(tech_result.loc[tech_result["Agreement"].idxmax()])
print("\nMOST DISAGREEMENT:")
print(tech_result.loc[tech_result["Disagreement"].idxmax()])

# ── Rating ───────────────────────────────────────────────────
print("\n── Rating ──")

rate_overall = []

for label in rate_labels:

    agreement    = 0
    disagreement = 0

    for _, row in merged.iterrows():

        vals  = [row["Rate_A1"], row["Rate_A2"], row["Rate_A3"]]
        count = vals.count(label)

        if count >= 2:
            agreement += 1
        elif count == 1:
            disagreement += 1

    rate_overall.append([label, agreement, disagreement])

rate_result = pd.DataFrame(
    rate_overall,
    columns=["Rating", "Agreement", "Disagreement"]
)

print(rate_result)
print("\nMOST AGREEMENT:")
print(rate_result.loc[rate_result["Agreement"].idxmax()])
print("\nMOST DISAGREEMENT:")
print(rate_result.loc[rate_result["Disagreement"].idxmax()])


# ============================================================
# PART 2: PAIRWISE AGREEMENT / DISAGREEMENT PER LABEL
# ============================================================

print("\n================================================")
print("PAIRWISE LABEL AGREEMENT / DISAGREEMENT")
print("================================================")

for a, b in pairs:

    print(f"\n====================")
    print(f"{a} vs {b}")
    print("====================")

    # ── Technique ────────────────────────────────────────────
    print("\n── Technique ──")

    tech_rows = []

    for label in tech_labels:

        agreement = (
            (merged[f"Tech_{a}"] == label) &
            (merged[f"Tech_{b}"] == label)
        ).sum()

        disagreement = (
            ((merged[f"Tech_{a}"] == label) &
             (merged[f"Tech_{b}"] != label))
            |
            ((merged[f"Tech_{a}"] != label) &
             (merged[f"Tech_{b}"] == label))
        ).sum()

        tech_rows.append([label, agreement, disagreement])

    print(pd.DataFrame(
        tech_rows,
        columns=["Technique", "Agreement", "Disagreement"]
    ))

    # ── Rating ───────────────────────────────────────────────
    print("\n── Rating ──")

    rate_rows = []

    for label in rate_labels:

        agreement = (
            (merged[f"Rate_{a}"] == label) &
            (merged[f"Rate_{b}"] == label)
        ).sum()

        disagreement = (
            ((merged[f"Rate_{a}"] == label) &
             (merged[f"Rate_{b}"] != label))
            |
            ((merged[f"Rate_{a}"] != label) &
             (merged[f"Rate_{b}"] == label))
        ).sum()

        rate_rows.append([label, agreement, disagreement])

    print(pd.DataFrame(
        rate_rows,
        columns=["Rating", "Agreement", "Disagreement"]
    ))


# ============================================================
# PART 3: RATIOS OF LABELS FOR EACH ANNOTATOR
# ============================================================

print("\n================================================")
print("LABEL RATIOS FOR EACH ANNOTATOR")
print("================================================")

for ann in annotators:

    print(f"\n── {ann} ──")

    print("Technique Ratios:")
    print(
        (merged[f"Tech_{ann}"]
         .value_counts(normalize=True) * 100)
        .round(2)
    )

    print("Rating Ratios:")
    print(
        (merged[f"Rate_{ann}"]
         .value_counts(normalize=True) * 100)
        .round(2)
    )


# ============================================================
# PART 4: AVERAGE RATIOS ACROSS ALL ANNOTATORS
# ============================================================

print("\n================================================")
print("AVERAGE RATIOS")
print("================================================")

all_tech = pd.concat([
    merged["Tech_A1"],
    merged["Tech_A2"],
    merged["Tech_A3"]
])

all_rate = pd.concat([
    merged["Rate_A1"],
    merged["Rate_A2"],
    merged["Rate_A3"]
])

print("\nAverage Technique Ratio:")
print(
    (all_tech.value_counts(normalize=True) * 100)
    .round(2)
)

print("\nAverage Rating Ratio:")
print(
    (all_rate.value_counts(normalize=True) * 100)
    .round(2)
)



OVERALL LABEL AGREEMENT / DISAGREEMENT

── Technique ──
  Technique  Agreement  Disagreement
0         A          1            11
1         I          2            10
2         M          2            23
3         S         44            31
4         W         85            18

MOST AGREEMENT:
Technique        W
Agreement       85
Disagreement    18
Name: 4, dtype: object

MOST DISAGREEMENT:
Technique        S
Agreement       44
Disagreement    31
Name: 3, dtype: object

── Rating ──
   Rating  Agreement  Disagreement
0       0         53            63
1       1         94            38

MOST AGREEMENT:
Rating           1
Agreement       94
Disagreement    38
Name: 1, dtype: int64

MOST DISAGREEMENT:
Rating           0
Agreement       53
Disagreement    63
Name: 0, dtype: int64

PAIRWISE LABEL AGREEMENT / DISAGREEMENT

A1 vs A2

── Technique ──
  Technique  Agreement  Disagreement
0         A          0             9
1         I          1             2
2         M          0         

In [ ]:
"""
Cell 3 — Annotation Count Table and Ratio Table

Depends on: merged, tech_labels, rate_labels, annotators (defined in Cell 0).

Produces count and ratio tables for both Technique and Rating label sets,
making it easy to compare how much each annotator used each label in raw
counts and as percentages.

Parts:
  1. TECHNIQUE COUNTS   : Table (rows = tech labels, cols = A1/A2/A3/Average)
                          with raw usage counts per annotator.
  2. TECHNIQUE RATIOS   : Same table converted to percentages, with Average %.
  3. RATING COUNTS      : Table (rows = rating labels, cols = A1/A2/A3/Average)
                          with raw usage counts per annotator.
  4. RATING RATIOS      : Same table converted to percentages, with Average %.
"""

# ============================================================
# COUNT + RATIO TABLES — Technique + Rating
# ============================================================

# merged, tech_labels, rate_labels, annotators already defined in Cell 0

import pandas as pd

# ============================================================
# PART 1: TECHNIQUE COUNTS
# ============================================================
label_info={'A':'Absurdity', 'W':'Wordplay', 'S':'Surprise', 'I':'Irony', 'M':'Miscellaneous'}
print(label_info)
print("\n================================================")
print("TECHNIQUE ANNOTATION COUNTS")
print("================================================")

tech_table = pd.DataFrame(index=tech_labels)

for ann in annotators:

    counts = merged[f"Tech_{ann}"].value_counts()

    tech_table[ann] = [
        counts.get(label, 0)
        for label in tech_labels
    ]

tech_table["Average"] = tech_table.mean(axis=1).round(2)

print(tech_table)


# ============================================================
# PART 2: TECHNIQUE RATIOS (%)
# ============================================================

print("\n================================================")
print("TECHNIQUE RATIOS (%)")
print("================================================")

tech_ratio = tech_table.copy()

for col in ["A1", "A2", "A3"]:
    total = tech_ratio[col].sum()

    if total != 0:
        tech_ratio[col] = (tech_ratio[col] / total * 100).round(2)
    else:
        tech_ratio[col] = 0

tech_ratio["Average %"] = (
    tech_ratio[["A1", "A2", "A3"]].mean(axis=1)
).round(2)

print(tech_ratio)


# ============================================================
# PART 3: RATING COUNTS
# ============================================================

print("\n================================================")
print("RATING ANNOTATION COUNTS")
print("================================================")

rate_table = pd.DataFrame(index=rate_labels)

for ann in annotators:

    counts = merged[f"Rate_{ann}"].value_counts()

    rate_table[ann] = [
        counts.get(label, 0)
        for label in rate_labels
    ]

rate_table["Average"] = rate_table.mean(axis=1).round(2)

print(rate_table)


# ============================================================
# PART 4: RATING RATIOS (%)
# ============================================================

print("\n================================================")
print("RATING RATIOS (%)")
print("================================================")

rate_ratio = rate_table.copy()

for col in ["A1", "A2", "A3"]:
    total = rate_ratio[col].sum()

    if total != 0:
        rate_ratio[col] = (rate_ratio[col] / total * 100).round(2)
    else:
        rate_ratio[col] = 0

rate_ratio["Average %"] = (
    rate_ratio[["A1", "A2", "A3"]].mean(axis=1)
).round(2)

print(rate_ratio)



TECHNIQUE ANNOTATION COUNTS
   A1  A2  A3  Average
A   3   6   4     4.33
I   3   1  10     4.67
M   7   0  20     9.00
S  56  50  26    44.00
W  78  90  87    85.00

TECHNIQUE RATIOS (%)
      A1     A2     A3  Average  Average %
A   2.04   4.08   2.72     4.33       2.95
I   2.04   0.68   6.80     4.67       3.17
M   4.76   0.00  13.61     9.00       6.12
S  38.10  34.01  17.69    44.00      29.93
W  53.06  61.22  59.18    85.00      57.82

RATING ANNOTATION COUNTS
   A1  A2  A3  Average
0  69  63  52    61.33
1  78  84  95    85.67

RATING RATIOS (%)
      A1     A2     A3  Average  Average %
0  46.94  42.86  35.37    61.33      41.72
1  53.06  57.14  64.63    85.67      58.28


In [ ]:
"""
Cell 4 — Krippendorff's Alpha Per Label via Binary Transformation

Depends on: df1, df2, df3 (loaded in Cell 0).

Computes a per-label Krippendorff's Alpha by binarising annotations: for
each label, every annotation is converted to 1 (annotator used this label)
or 0 (annotator did not). This gives a reliability score specific to each
label rather than a single score across all labels. Runs for both Technique
and Rating label sets.

Note: this cell creates its own merged DataFrame ('merged_bin') with plain
column names Tech_A1/Tech_A2/Tech_A3 and Rate_A1/Rate_A2/Rate_A3 — it
uses the already-cleaned df1/df2/df3 from Cell 0 so no re-cleaning is needed.

Steps:
  MERGE             : Inner-join the three DataFrames on 'Jokes' into
                      merged_bin with both label columns per annotator.
  to_binary()       : Helper that converts a label series to a binary list
                      for a given target label.
  TECHNIQUE — OVERALL ALPHA : Three-way nominal alpha per Technique label.
  TECHNIQUE — PAIRWISE ALPHA: Pairwise nominal alpha per Technique label
                      (5 labels × 3 pairs = 15 values).
  RATING — OVERALL ALPHA    : Three-way ordinal alpha per Rating label.
  RATING — PAIRWISE ALPHA   : Pairwise ordinal alpha per Rating label
                      (2 labels × 3 pairs = 6 values).

Produces:
  merged_bin              — binary-ready merged DataFrame
  tech_overall_results    — list of [label, alpha] for Technique three-way
  tech_pairwise_results   — list of [label, p1, p2, alpha] for Technique pairwise
  rate_overall_results    — list of [label, alpha] for Rating three-way
  rate_pairwise_results   — list of [label, p1, p2, alpha] for Rating pairwise
"""

# ============================================================
# KRIPPENDORFF'S ALPHA PER LABEL (BINARY TRANSFORMATION)
# Technique (nominal) + Rating (ordinal)
# ============================================================

# df1, df2, df3 already loaded and cleaned in Cell 0

import pandas as pd
import krippendorff

# ------------------------------------------------------------
# MERGE (by Jokes)
# ------------------------------------------------------------

merged_bin = df1.rename(columns={
    'Technique': 'Tech_A1',
    'Rating':    'Rate_A1'
})

merged_bin = merged_bin.merge(
    df2.rename(columns={'Technique': 'Tech_A2', 'Rating': 'Rate_A2'}),
    on='Jokes'
)

merged_bin = merged_bin.merge(
    df3.rename(columns={'Technique': 'Tech_A3', 'Rating': 'Rate_A3'}),
    on='Jokes'
)

# ------------------------------------------------------------
# FUNCTION: BINARY TRANSFORMATION
# ------------------------------------------------------------

def to_binary(series, target_label):
    return [1 if x == target_label else 0 for x in series]

# ------------------------------------------------------------
# TECHNIQUE — OVERALL ALPHA PER LABEL
# ------------------------------------------------------------

print("\n======================================")
print("TECHNIQUE — KRIPPENDORFF'S ALPHA PER LABEL (OVERALL)")
print("======================================")
label_info={'A':'Absurdity', 'W':'Wordplay', 'S':'Surprise', 'I':'Irony', 'M':'Miscellaneous'}
print(label_info)
tech_overall_results = []

for label in tech_labels:

    data = [
        to_binary(merged_bin['Tech_A1'], label),
        to_binary(merged_bin['Tech_A2'], label),
        to_binary(merged_bin['Tech_A3'], label)
    ]

    alpha = krippendorff.alpha(
        reliability_data=data,
        level_of_measurement='nominal'
    )

    print(f"{label}: Alpha = {round(alpha, 4)}")

    tech_overall_results.append([label, round(alpha, 4)])


# ------------------------------------------------------------
# TECHNIQUE — PAIRWISE ALPHA PER LABEL
# ------------------------------------------------------------

print("\n======================================")
print("TECHNIQUE — PAIRWISE ALPHA PER LABEL")
print("======================================")

bin_pairs = [('A1','A2'), ('A1','A3'), ('A2','A3')]

tech_pairwise_results = []

for label in tech_labels:

    print(f"\n--- Technique Label: {label} ---")

    for p1, p2 in bin_pairs:

        data = [
            to_binary(merged_bin[f"Tech_{p1}"], label),
            to_binary(merged_bin[f"Tech_{p2}"], label)
        ]

        alpha = krippendorff.alpha(
            reliability_data=data,
            level_of_measurement='nominal'
        )

        print(f"  {p1}-{p2}: {round(alpha, 4)}")

        tech_pairwise_results.append([label, p1, p2, round(alpha, 4)])


# ------------------------------------------------------------
# RATING — OVERALL ALPHA PER LABEL
# ------------------------------------------------------------

print("\n======================================")
print("RATING — KRIPPENDORFF'S ALPHA PER LABEL (OVERALL)")
print("======================================")

rate_overall_results = []

for label in rate_labels:

    data = [
        to_binary(merged_bin['Rate_A1'], label),
        to_binary(merged_bin['Rate_A2'], label),
        to_binary(merged_bin['Rate_A3'], label)
    ]

    alpha = krippendorff.alpha(
        reliability_data=data,
        level_of_measurement='ordinal'
    )

    print(f"{label}: Alpha = {round(alpha, 4)}")

    rate_overall_results.append([label, round(alpha, 4)])


# ------------------------------------------------------------
# RATING — PAIRWISE ALPHA PER LABEL
# ------------------------------------------------------------

print("\n======================================")
print("RATING — PAIRWISE ALPHA PER LABEL")
print("======================================")

rate_pairwise_results = []

for label in rate_labels:

    print(f"\n--- Rating Label: {label} ---")

    for p1, p2 in bin_pairs:

        data = [
            to_binary(merged_bin[f"Rate_{p1}"], label),
            to_binary(merged_bin[f"Rate_{p2}"], label)
        ]

        alpha = krippendorff.alpha(
            reliability_data=data,
            level_of_measurement='ordinal'
        )

        print(f"  {p1}-{p2}: {round(alpha, 4)}")

        rate_pairwise_results.append([label, p1, p2, round(alpha, 4)])



TECHNIQUE — KRIPPENDORFF'S ALPHA PER LABEL (OVERALL)
A: Alpha = 0.051
I: Alpha = 0.1168
M: Alpha = 0.0159
S: Alpha = 0.3312
W: Alpha = 0.666

TECHNIQUE — PAIRWISE ALPHA PER LABEL

--- Technique Label: A ---
  A1-A2: -0.0281
  A1-A3: -0.0209
  A2-A3: 0.1746

--- Technique Label: I ---
  A1-A2: 0.4948
  A1-A3: 0.1177
  A2-A3: -0.0353

--- Technique Label: M ---
  A1-A2: -0.0209
  A1-A3: 0.0652
  A2-A3: -0.0693

--- Technique Label: S ---
  A1-A2: 0.6177
  A1-A3: 0.1236
  A2-A3: 0.1865

--- Technique Label: W ---
  A1-A2: 0.7232
  A1-A3: 0.6283
  A2-A3: 0.6463

RATING — KRIPPENDORFF'S ALPHA PER LABEL (OVERALL)
0: Alpha = 0.0602
1: Alpha = 0.0602

RATING — PAIRWISE ALPHA PER LABEL

--- Rating Label: 0 ---
  A1-A2: 0.1231
  A1-A3: 0.0062
  A2-A3: 0.0463

--- Rating Label: 1 ---
  A1-A2: 0.1231
  A1-A3: 0.0062
  A2-A3: 0.0463
